# Singing-finetuned-DAC — before/after comparison

Compare reconstruction of the **pretrained DAC 24 kHz** vs the **singing fine-tuned** model
on your own audio: listen to both, view mel-spectrograms, and print quality metrics.

- Pretrained weights: downloaded automatically by `dac`.
- Fine-tuned weights: pulled from the Hugging Face Hub (`Joshua-1995/Singing-finetuned-DAC`).


## 1. Setup
On a fresh machine / Colab, install the deps. For a **Blackwell (sm_120) GPU** install the
CUDA 12.8 PyTorch build separately (see the repo README); on Colab the default torch is fine.

In [ ]:
%pip install -q descript-audio-codec librosa matplotlib huggingface_hub
# Blackwell GPU only:
# %pip install -q torch torchaudio --index-url https://download.pytorch.org/whl/cu128

In [ ]:
import torch, numpy as np, librosa, librosa.display
import matplotlib.pyplot as plt
import dac
from huggingface_hub import hf_hub_download
from IPython.display import Audio, display

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

## 2. Load both models

In [ ]:
# Pretrained DAC 24 kHz (official)
pre = dac.DAC.load(str(dac.utils.download(model_type='24khz'))).to(device).eval()

# Fine-tuned (singing) — single-file generator from the Hugging Face Hub
ft_path = hf_hub_download('Joshua-1995/Singing-finetuned-DAC', 'dac_singing_finetune_24khz.pth')
ft = dac.DAC.load(ft_path).to(device).eval()
print('loaded: pretrained + fine-tuned')

## 3. Choose an audio file
Set `AUDIO_PATH` to any singing clip (wav/flac/mp3; mono or stereo, any sample rate — it is
resampled to 24 kHz). A short 5–10 s excerpt is easiest to compare by ear.

In [ ]:
AUDIO_PATH = 'your_song.wav'   # <-- change me
MAX_SECONDS = 10               # set None to use the whole file

wav, _ = librosa.load(AUDIO_PATH, sr=24000, mono=True)
if MAX_SECONDS:
    wav = wav[: int(24000 * MAX_SECONDS)]
print('duration: %.1f s' % (len(wav) / 24000))

## 4. Encode → decode with both models

In [ ]:
@torch.no_grad()
def reconstruct(model, wav, sr=24000):
    x = torch.from_numpy(wav.astype('float32'))[None, None].to(device)
    z, *_ = model.encode(model.preprocess(x, sr))
    y = model.decode(z)
    n = min(y.shape[-1], x.shape[-1])
    return y[0, 0, :n].cpu().numpy()

rec_pre = reconstruct(pre, wav)
rec_ft  = reconstruct(ft,  wav)
print('done')

## 5. Listen (original vs pretrained vs fine-tuned)

In [ ]:
print('Original');             display(Audio(wav,     rate=24000))
print('Pretrained DAC');        display(Audio(rec_pre, rate=24000))
print('Fine-tuned (singing)');  display(Audio(rec_ft,  rate=24000))

## 6. Mel-spectrograms

In [ ]:
def mel(w):
    S = librosa.feature.melspectrogram(y=w, sr=24000, n_fft=1024, hop_length=256, n_mels=128, fmax=12000)
    return librosa.power_to_db(S, ref=np.max)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, (w, t) in zip(axes, [(wav,'Original'), (rec_pre,'Pretrained DAC'), (rec_ft,'Fine-tuned (singing)')]):
    librosa.display.specshow(mel(w), sr=24000, hop_length=256, x_axis='time', y_axis='mel',
                             fmax=12000, ax=ax, vmin=-80, vmax=0, cmap='magma')
    ax.set_title(t)
plt.tight_layout(); plt.show()

## 7. Quick metrics (this clip)
SI-SDR (waveform fidelity, dB ↑) and mel distance (↓). Note: single-clip numbers are
illustrative; the repo's aggregate eval is over 160 clips.

In [ ]:
def si_sdr(ref, est, eps=1e-8):
    ref = ref - ref.mean(); est = est - est.mean()
    a = (est * ref).sum() / ((ref ** 2).sum() + eps)
    t = a * ref; n = est - t
    return 10 * np.log10(((t ** 2).sum() + eps) / ((n ** 2).sum() + eps))

n = min(len(wav), len(rec_pre), len(rec_ft))
print('SI-SDR (dB)   pretrained: %+.2f   |   fine-tuned: %+.2f' %
      (si_sdr(wav[:n], rec_pre[:n]), si_sdr(wav[:n], rec_ft[:n])))

def mel_l1(a, b):
    n = min(len(a), len(b))
    return float(np.mean(np.abs(mel(a[:n]) - mel(b[:n]))))
print('mel L1 (dB)   pretrained: %.3f   |   fine-tuned: %.3f' %
      (mel_l1(wav, rec_pre), mel_l1(wav, rec_ft)))